# Chronic Kidney Disease: Data Audit and Cleaning

This notebook performs the initial data audit and cleaning workflow for the **Interpretable AI for Chronic Kidney Disease Detection** project.

## Objectives

- Load the raw CKD dataset
- Inspect dataset structure and variable types
- Identify missing values and inconsistent formatting
- Standardize column names
- Clean and prepare fields for downstream analysis
- Save a processed version of the dataset for later notebooks

## Dataset

This project uses the Chronic Kidney Disease dataset from Kaggle:

https://www.kaggle.com/datasets/mansoordaku/ckdisease

The dataset contains clinical measurements and patient indicators commonly used in CKD diagnosis, including blood chemistry values, urinalysis indicators, and comorbid conditions.

In [2]:
# ============================================================
# 1. Imports and display settings
# ============================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# ============================================================
# Load dataset
# ============================================================

FILE_PATH = "/content/ckd-interpretable-ai/data/ckd-dataset-v2.csv"

import pandas as pd

df_raw = pd.read_csv(FILE_PATH)
df = df_raw.copy()

print("Dataset loaded successfully.")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

df.head()

Dataset loaded successfully.
Shape: 202 rows x 29 columns


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,class,meta
2,0,0,1.019 - 1.021,1 - 1,ckd,0,< 0,0,0,0,< 112,< 48.1,138 - 143,< 3.65,< 7.31,11.3 - 12.6,33.5 - 37.4,4.46 - 5.05,7360 - 9740,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
3,0,0,1.009 - 1.011,< 0,ckd,0,< 0,0,0,0,112 - 154,< 48.1,133 - 138,< 3.65,< 7.31,11.3 - 12.6,33.5 - 37.4,4.46 - 5.05,12120 - 14500,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
4,0,0,1.009 - 1.011,≥ 4,ckd,1,< 0,1,0,1,< 112,48.1 - 86.2,133 - 138,< 3.65,< 7.31,8.7 - 10,29.6 - 33.5,4.46 - 5.05,14500 - 16880,0,0,0,1,0,0,127.281 - 152.446,s1,1,< 12


## Remove Non-Data Metadata Rows

The CKD dataset contains two rows at the top that are not actual patient observations.

Row 0 contains variable type descriptors such as "discrete".  
Row 1 is an empty row.

These rows are removed to ensure the dataset contains only valid observations.

In [5]:
# ============================================================
# Remove metadata rows
# ============================================================

df = df.iloc[2:].reset_index(drop=True)

print("Metadata rows removed.")
print(f"New dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

df.head()

Metadata rows removed.
New dataset shape: 200 rows x 29 columns


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,0,0,1.019 - 1.021,1 - 1,ckd,0,< 0,0,0,0,< 112,< 48.1,138 - 143,< 3.65,< 7.31,11.3 - 12.6,33.5 - 37.4,4.46 - 5.05,7360 - 9740,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
1,0,0,1.009 - 1.011,< 0,ckd,0,< 0,0,0,0,112 - 154,< 48.1,133 - 138,< 3.65,< 7.31,11.3 - 12.6,33.5 - 37.4,4.46 - 5.05,12120 - 14500,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
2,0,0,1.009 - 1.011,≥ 4,ckd,1,< 0,1,0,1,< 112,48.1 - 86.2,133 - 138,< 3.65,< 7.31,8.7 - 10,29.6 - 33.5,4.46 - 5.05,14500 - 16880,0,0,0,1,0,0,127.281 - 152.446,s1,1,< 12
3,1,1,1.009 - 1.011,3 - 3,ckd,0,< 0,0,0,0,112 - 154,< 48.1,133 - 138,< 3.65,< 7.31,13.9 - 15.2,41.3 - 45.2,4.46 - 5.05,7360 - 9740,0,0,0,0,0,0,127.281 - 152.446,s1,1,< 12
4,0,0,1.015 - 1.017,< 0,ckd,0,< 0,0,0,0,154 - 196,< 48.1,133 - 138,< 3.65,< 7.31,13.9 - 15.2,37.4 - 41.3,5.05 - 5.64,7360 - 9740,0,1,0,1,1,0,127.281 - 152.446,s1,1,12 - 20


## Dataset Audit

Before cleaning or transforming the dataset, we perform a data audit to understand:

- Column data types
- Missing values
- Unique values
- Sample entries

This helps identify issues such as:

- numerical values stored as strings
- ranges and inequality expressions
- categorical variables
- potential missing data

In [6]:
# ============================================================
# Dataset Audit Report
# ============================================================

audit_results = []

for col in df.columns:

    missing = df[col].isna().sum()
    unique = df[col].nunique(dropna=True)
    dtype = df[col].dtype

    sample_values = df[col].dropna().astype(str).unique()[:5]

    audit_results.append({
        "column": col,
        "dtype": dtype,
        "missing_values": missing,
        "unique_values": unique,
        "sample_values": ", ".join(sample_values)
    })

audit_df = pd.DataFrame(audit_results)

audit_df.sort_values(by="unique_values", ascending=False)

,column,dtype,missing_values,unique_values,sample_values
25,grf,object,0,11,"≥ 227.944, 127.281 - 152.446, 102.115 - 127.281, 177.612 - 202.778, 26.6175 - 51.7832"
15,hemo,object,0,10,"11.3 - 12.6, 8.7 - 10, 13.9 - 15.2, ≥ 16.5, 10 - 11.3"
28,age,object,0,10,"< 12, 12 - 20, 20 - 27, 27 - 35, 35 - 43"
16,pcv,object,0,10,"33.5 - 37.4, 29.6 - 33.5, 41.3 - 45.2, 37.4 - 41.3, ≥ 49.1"
10,bgr,object,0,10,"< 112, 112 - 154, 154 - 196, 406 - 448, 238 - 280"
12,sod,object,0,9,"138 - 143, 133 - 138, 123 - 128, 143 - 148, 148 - 153"
18,wbcc,object,0,9,"7360 - 9740, 12120 - 14500, 14500 - 16880, 4980 - 7360, < 4980"
17,rbcc,object,0,9,"4.46 - 5.05, 5.05 - 5.64, 3.28 - 3.87, 3.87 - 4.46, 6.23 - 6.82"
11,bu,object,0,8,"< 48.1, 48.1 - 86.2, 200.5 - 238.6, 124.3 - 162.4, 86.2 - 124.3"
13,sc,object,0,7,"< 3.65, 3.65 - 6.8, 16.25 - 19.4, 6.8 - 9.95, 13.1 - 16.25"


## Converting Range-Based Medical Values

Many variables in the CKD dataset are represented as ranges or inequalities rather than single values.

Examples include:

- `1.019 - 1.021`
- `<112`
- `≥4`
- `48.1 - 86.2`

To enable machine learning models to use these features, these values must be converted into numeric representations.

The following rules are applied:

| Expression | Conversion |
|------|------|
| `a - b` | midpoint `(a + b) / 2` |
| `< x` | value `x` |
| `> x` or `≥ x` | value `x` |
| single value | converted directly |

This preserves the approximate magnitude of each measurement while allowing numerical modeling.

In [7]:
# ============================================================
# Convert range / inequality values to numeric
# ============================================================

import re

def convert_medical_value(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    # range: "a - b"
    if "-" in value:
        parts = re.findall(r"[\d\.]+", value)
        if len(parts) == 2:
            return (float(parts[0]) + float(parts[1])) / 2

    # inequality < or >
    if "<" in value or ">" in value or "≥" in value or "≤" in value:
        num = re.findall(r"[\d\.]+", value)
        if num:
            return float(num[0])

    # normal numeric
    try:
        return float(value)
    except:
        return value

df_numeric = df.applymap(convert_medical_value)

df_numeric.head()

/tmp/ipykernel_411/2261179445.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_numeric = df.applymap(convert_medical_value)


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,0.0,0.0,1.020,1.0,ckd,0.0,0.0,0.0,0.0,0.0,112.0,48.10,140.5,3.65,7.31,11.95,35.45,4.755,8550.0,0.0,0.0,0.0,0.0,0.0,0.0,227.944,s1,1.0,12.0
1,0.0,0.0,1.010,0.0,ckd,0.0,0.0,0.0,0.0,0.0,133.0,48.10,135.5,3.65,7.31,11.95,35.45,4.755,13310.0,0.0,0.0,0.0,0.0,0.0,0.0,227.944,s1,1.0,12.0
2,0.0,0.0,1.010,4.0,ckd,1.0,0.0,1.0,0.0,1.0,112.0,67.15,135.5,3.65,7.31,9.35,31.55,4.755,15690.0,0.0,0.0,0.0,1.0,0.0,0.0,139.8635,s1,1.0,12.0
3,1.0,1.0,1.010,3.0,ckd,0.0,0.0,0.0,0.0,0.0,133.0,48.10,135.5,3.65,7.31,14.55,43.25,4.755,8550.0,0.0,0.0,0.0,0.0,0.0,0.0,139.8635,s1,1.0,12.0
4,0.0,0.0,1.016,0.0,ckd,0.0,0.0,0.0,0.0,0.0,175.0,48.10,135.5,3.65,7.31,14.55,39.35,5.345,8550.0,0.0,1.0,0.0,1.0,1.0,0.0,139.8635,s1,1.0,16.0


## Encode Target Variable

The `class` column represents the target variable indicating whether a patient has chronic kidney disease.

To prepare the dataset for machine learning models, the labels are encoded as:

| Original | Encoded |
|--------|--------|
| ckd | 1 |
| notckd | 0 |

In [8]:
# ============================================================
# Encode target variable
# ============================================================

df_numeric["class"] = df_numeric["class"].map({
    "ckd": 1,
    "notckd": 0
})

print("Target variable encoded.")
df_numeric["class"].value_counts()

Target variable encoded.


,count
class,
1,128
0,72


## Remove Data Leakage Features

Some variables in the dataset may directly encode the disease status rather than independent predictors.

Including such variables could artificially inflate model performance.

The following columns are removed to prevent data leakage:

- `stage`
- `affected`

In [9]:
# ============================================================
# Remove leakage columns
# ============================================================

leakage_columns = ["stage", "affected"]

df_clean = df_numeric.drop(columns=leakage_columns)

print("Columns removed:", leakage_columns)
print("New dataset shape:", df_clean.shape)

df_clean.head()

Columns removed: ['stage', 'affected']
New dataset shape: (200, 27)


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,age
0,0.0,0.0,1.020,1.0,1,0.0,0.0,0.0,0.0,0.0,112.0,48.10,140.5,3.65,7.31,11.95,35.45,4.755,8550.0,0.0,0.0,0.0,0.0,0.0,0.0,227.944,12.0
1,0.0,0.0,1.010,0.0,1,0.0,0.0,0.0,0.0,0.0,133.0,48.10,135.5,3.65,7.31,11.95,35.45,4.755,13310.0,0.0,0.0,0.0,0.0,0.0,0.0,227.944,12.0
2,0.0,0.0,1.010,4.0,1,1.0,0.0,1.0,0.0,1.0,112.0,67.15,135.5,3.65,7.31,9.35,31.55,4.755,15690.0,0.0,0.0,0.0,1.0,0.0,0.0,139.8635,12.0
3,1.0,1.0,1.010,3.0,1,0.0,0.0,0.0,0.0,0.0,133.0,48.10,135.5,3.65,7.31,14.55,43.25,4.755,8550.0,0.0,0.0,0.0,0.0,0.0,0.0,139.8635,12.0
4,0.0,0.0,1.016,0.0,1,0.0,0.0,0.0,0.0,0.0,175.0,48.10,135.5,3.65,7.31,14.55,39.35,5.345,8550.0,0.0,1.0,0.0,1.0,1.0,0.0,139.8635,16.0


In [10]:
df_clean.to_csv("ckd_cleaned.csv", index=False)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.
